<a href="https://colab.research.google.com/github/FaisalBinAbbas321/SWE485-Project/blob/main/Phase4_Generative_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phase 4 — Generative AI
        
**Phase Overview **

## 1. Phase Overview

In this phase, we integrate a **Generative AI model**  into our stroke-risk advice system.

The goals are:

- Use a Generative AI model to **generate detailed advice/explanations** based on user input (patient information / risk factors).
- Implement **at least two different prompt templates** and compare their outputs.
- Analyze the **quality, detail, and relevance** of the responses.
- Choose a **final prompt template** and justify why it is most suitable.

This notebook focuses on:

1. **Implementation:** Calling a Generative AI model via API (here we demonstrate using GPT).
2. **Template Comparison & Analysis:** Comparing two prompt templates with multiple example inputs.
3. **Justification:** Selecting and justifying the final template for our system.

In [1]:

!pip install --quiet openai

import os
import json
from datetime import datetime

from openai import OpenAI


## 2. API Setup – Connecting to a Generative AI Model

In this notebook, we demonstrate integration with a **GPT-style model** using the OpenAI API.

In [22]:


os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"




In [14]:
api_key = os.getenv("OPENAI_API_KEY")

if api_key is None or api_key == "YOUR_API_KEY_HERE":
    raise ValueError(
        "OPENAI_API_KEY is not set or still uses the placeholder.\n"
        "When running this notebook in Colab, set:\n"
        "  os.environ['OPENAI_API_KEY'] = 'sk-REAL_KEY_HERE'\n"

    )


client = OpenAI(api_key=api_key)


GEN_MODEL = "gpt-4.1-mini"


## 3. Use Case – Generative Advice for Stroke Risk

Our system predicts stroke risk (from previous phases) and we now want to:

- Explain the risk in natural language.
- Provide **simple, personalized advice** for patients.
- Use a **Generative AI model** to produce this advice based on a patient profile.

In this notebook:

- We define example **patient profiles** as Python dictionaries (age, BMI, etc.).
- We design **two prompt templates**:
  - Template 1: Formal, clinical style.
  - Template 2: Friendly, patient-focused style.
- We apply both templates to the same patients and compare the outputs.


In [15]:


patient_A = {
    "id": 1,
    "age": 62,
    "gender": "Female",
    "hypertension": 1,
    "heart_disease": 0,
    "ever_married": "Yes",
    "work_type": "Private",
    "Residence_type": "Urban",
    "avg_glucose_level": 210.5,
    "bmi": 31.2,
    "smoking_status": "formerly smoked",
    "predicted_stroke_risk": "High"
}

patient_B = {
    "id": 2,
    "age": 35,
    "gender": "Male",
    "hypertension": 0,
    "heart_disease": 0,
    "ever_married": "No",
    "work_type": "Self-employed",
    "Residence_type": "Rural",
    "avg_glucose_level": 90.0,
    "bmi": 23.5,
    "smoking_status": "never smoked",
    "predicted_stroke_risk": "Low"
}

example_patients = [patient_A, patient_B]

example_patients


[{'id': 1,
  'age': 62,
  'gender': 'Female',
  'hypertension': 1,
  'heart_disease': 0,
  'ever_married': 'Yes',
  'work_type': 'Private',
  'Residence_type': 'Urban',
  'avg_glucose_level': 210.5,
  'bmi': 31.2,
  'smoking_status': 'formerly smoked',
  'predicted_stroke_risk': 'High'},
 {'id': 2,
  'age': 35,
  'gender': 'Male',
  'hypertension': 0,
  'heart_disease': 0,
  'ever_married': 'No',
  'work_type': 'Self-employed',
  'Residence_type': 'Rural',
  'avg_glucose_level': 90.0,
  'bmi': 23.5,
  'smoking_status': 'never smoked',
  'predicted_stroke_risk': 'Low'}]

## 4. Prompt Templates

We design **two different prompt templates** that instruct the Generative AI:

- **Template 1 – Clinical / Detailed Style**
  - More formal, structured, and technical.
  - Suitable as a "report-like" explanation.

- **Template 2 – Friendly / Patient-Focused Style**
  - Simpler wording, bullet points.
  - Intended for non-expert users.

We will later compare:

- Clarity
- Level of detail
- Relevance
- Suitability for our target users


In [16]:
template_1 = """
You are an AI medical assistant helping with stroke risk explanation.

The following is a patient profile from a stroke prediction system:

{patient_json}

Task:
1. Provide a formal, clinically oriented summary of the patient's possible stroke risk.
2. Explain which risk factors are most significant and why (age, hypertension, glucose, BMI, smoking, etc.).
3. Provide detailed recommendations for lifestyle changes and medical follow-up.
4. Use a professional, clear tone suitable for a medical-style report, but still understandable by an educated patient.
5. Do NOT invent any lab results, diagnoses, or treatments that are not directly implied by the information.
6. Always remind the user that this information is general and they must consult a doctor for personal medical advice.
"""

template_2 = """
You are a friendly health coach helping people understand stroke risk.

Here is the patient's information:

{patient_json}

Task:
1. Explain their situation in very simple, friendly language.
2. Avoid complex medical terms when possible. If you use a medical term, briefly explain it.
3. Use short paragraphs and bullet points.
4. Organize the advice under headings:
   - "What this means"
   - "What you can do"
   - "When to talk to a doctor"
5. Encourage the patient without causing unnecessary fear.
6. Make it very clear that this is not a diagnosis and they must talk to a doctor for medical decisions.
"""


In [17]:
def generate_advice(patient, template, model=GEN_MODEL):
    """
    Call the Generative AI model using a given template and patient data.
    Returns the generated advice text as a string.
    """
    patient_json = json.dumps(patient, indent=2)
    prompt = template.format(patient_json=patient_json)

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a careful and responsible AI health assistant. "
                    "You NEVER give definitive diagnoses, and you always remind users "
                    "that they must consult a doctor for personal medical advice."
                )
            },
            {"role": "user", "content": prompt}
        ],
        temperature=0.7,
        max_tokens=600
    )

    return response.choices[0].message.content.strip()


## 5. Running Both Templates on Example Patients

We now:

- Apply **Template 1 (clinical style)** and **Template 2 (friendly style)** to:
  - Patient A (high risk)
  - Patient B (low risk)
- Print the outputs side-by-side to visually compare:
  - Tone
  - Level of detail
  - Clarity
  - Relevance to the input profile

We will store these outputs in a list for later analysis.


In [ ]:
results = []

for patient in example_patients:
    print(f"==================== Patient ID: {patient['id']} | Risk: {patient['predicted_stroke_risk']} ====================\n")

    advice_t1 = generate_advice(patient, template_1)
    advice_t2 = generate_advice(patient, template_2)

    print("---- TEMPLATE 1 (Clinical Style) ----\n")
    print(advice_t1)
    print("\n---- TEMPLATE 2 (Friendly Style) ----\n")
    print(advice_t2)
    print("\n" + "="*100 + "\n")

    results.append({
        "patient_id": patient["id"],
        "risk_level": patient["predicted_stroke_risk"],
        "advice_template_1": advice_t1,
        "advice_template_2": advice_t2,
        "timestamp": datetime.now().isoformat()
    })


## 6. Template Comparison & Analysis

Using the generated outputs for Patient A and Patient B, we compare the two templates:

### 6.1. Style and Tone

- **Template 1 – Clinical Style:**
  - More formal, technical language.
  - Looks like a mini medical report.
  - May mention concepts like "risk factors", "cardiovascular risk", etc.

- **Template 2 – Friendly Style:**
  - Simple, conversational language.
  - Uses headings and bullet points.
  - Easier to read quickly, especially for non-experts.

### 6.2. Level of Detail

- Template 1:
  - Often explains *why* each factor matters in more detail.
  - Good for users who want a deeper explanation.

- Template 2:
  - Focuses more on **practical advice** and what the patient can do.
  - Less technical, more focused on day-to-day actions.

### 6.3. Relevance and Clarity

- Both templates:
  - Refer to the main risk factors (age, hypertension, BMI, glucose, smoking).
  - Provide general recommendations like healthy diet, physical activity, and seeing a doctor.

- Template 2:
  - Its headings (e.g., "What this means", "What you can do") improve navigation.
  - Less risk of overwhelming the user with jargon.

### 6.4. Safety

- Both templates:
  - Explicitly state that this is **not a diagnosis**.
  - Encourage the user to consult a doctor.
- This is important for a responsible health-related system.


## 7. Justification – Final Prompt Template Selection

Based on our comparison, we select **Template 2 (Friendly, Patient-Focused)** as the **primary prompt** for integration into the system.

### Reasons:

1. **Target Audience:**
   - Our system is intended for general users who may not have medical backgrounds.
   - Template 2 uses simple language and explains any medical terms briefly.

2. **Usability:**
   - Clear headings ("What this means", "What you can do", "When to talk to a doctor").
   - Bullet points help users quickly find the most important information.

3. **Emotional Impact:**
   - The tone is encouraging and supportive.
   - Reduces unnecessary fear while still emphasizing the importance of medical follow-up.

4. **Alignment with System Goals:**
   - Our overall system (Phases 1–3) predicts risk and gives **actionable advice**.
   - Template 2 fits better with a recommendation-oriented system for end-users.

5. **Safety:**
   - The template consistently reminds users that:
     - This is not a diagnosis.
     - They must consult a doctor for personal decisions.

## 8. Integration Note
The chosen prompt template (Template 2) will be used inside the system to generate natural-language advice for users after the stroke risk prediction is computed. The AI receives the patient information and returns personalized guidance.


> **Conclusion:** Template 2 is chosen as the main prompt template for the system’s advice feature.  
> Template 1 can be kept as an optional “detailed report” view for more advanced users, if needed.
